In [ ]:
from src.config import DATA_ROOT, FIGURE_ROOT, SEQ_LENGTH, SAMPLE_EVERY
from pathlib import Path
for folder in ["", "shap", "embeddings"]:
    (Path(FIGURE_ROOT) / folder).mkdir(parents=True, exist_ok=True)


# SHAP analysis
Generate attributions, then run the plotting cells below using the same definition.


In [ ]:
import pickle
import shap
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import sys

from src.main_train_eval_multi import *

chosen_def = 'fall20' # fall20, nadir90, fall20nadir90
print("Chosen Definition:", chosen_def)

# read fnames
with open((DATA_ROOT + '/splits_5fold_all'), 'rb') as f:
    split = pickle.load(f)[0]

# load data
trail_name = 'pre_real_all'
fold_i = 0
real_time = True
history = True
record_ = True
machine_param = True

# DEVICE is imported from src.models.layers
print("Overwriten DEVICE:", DEVICE)

eval_dataset = IDHDataset(
    split["test_fnames"], 
    facility="all",
    real_time=real_time,
    history=history,
    record=record_,
    acute_interv=True,
    machine_param=machine_param
)

torch.cuda.empty_cache()
gc.collect()
del split

# construct model
model = PreRealAll(
    emb_size=192, # 48, 192, 768, 2048, 4096
    num_layers=2, # 1, 2, 12, 24, 32
    num_heads=12, # 12, 32
    # data related:
    in_channel=8, # signal data size, 4 if .ph, 8 if .ph+.mp
    pre_size=18, # pre-session data size, 18 for pure num, 768 for llm embed
    seq_length=SEQ_LENGTH,
    # other setting
    use_cls=False,
    late_fuse=False,
    seg_embed=False,
    one_def_only=chosen_def 
)
model_record_name = (DATA_ROOT + '/exp_res/{}/{}_model_{}.pt').format(trail_name, trail_name, fold_i)
model.load_state_dict(torch.load(model_record_name, map_location=torch.device('cpu')))
model.eval().to(DEVICE)

# organize fnames
subject_to_samples = dict()
with open((DATA_ROOT + '/splits_5fold_all'), 'rb') as f:
    split = pickle.load(f)[0]

for fn in split["test_fnames"]:
    subject = fn.split("_")[0]
    if subject_to_samples.get(subject) is None:
        subject_to_samples[subject] = list()
    subject_to_samples[subject].append(fn)

# form new dataset
num_per_sub = 50
new_fns = list()
for sub in subject_to_samples:
    new_fns = new_fns + np.random.choice(subject_to_samples[sub], num_per_sub).tolist()

new_eval_dataset = IDHDataset(
    new_fns, 
    facility="all",
    real_time=real_time,
    history=history,
    record=record_,
    acute_interv=True,
    machine_param=machine_param
)
print("Subset length:", len(new_eval_dataset))

# main
# init
data_loader = DataLoader(new_eval_dataset, batch_size=128, shuffle=False, drop_last=True)

check_class = 1
explainer = None
first_batch = None
shap_values_accumulated = [list(), list(), list()]

num_batches = 0
# iterate over batch
for data_pack in tqdm(data_loader):
    # init explainer
    if explainer is None:
        first_batch = {
            'bodies': list(),
            'summaries': list(),
            'ehrs': list()
        }

        explainer = shap.GradientExplainer(model, [
            data_pack['bodies'].to(DEVICE),
            data_pack['summaries'].to(DEVICE),
            data_pack['decays'].to(DEVICE),
            data_pack['ehrs'].to(DEVICE),
            data_pack['ehr_decays'].to(DEVICE)
        ])
    
    # Calculate SHAP values for the current batch.
    shap_values_batch = explainer.shap_values([
        data_pack['bodies'].to(DEVICE),
        data_pack['summaries'].to(DEVICE),
        data_pack['decays'].to(DEVICE),
        data_pack['ehrs'].to(DEVICE),
        data_pack['ehr_decays'].to(DEVICE)
    ])

    # accumulate SHAP values
    shap_values_accumulated[0].append(shap_values_batch[0].astype(np.float16)) # real time
    shap_values_accumulated[1].append(shap_values_batch[1].astype(np.float16)) # summary
    shap_values_accumulated[2].append(shap_values_batch[3].astype(np.float16)) # ehrs

    first_batch["bodies"].append(data_pack['bodies'].cpu().numpy().astype(np.float16))
    first_batch["summaries"].append(data_pack['summaries'].cpu().numpy().astype(np.float16))
    first_batch["ehrs"].append(data_pack['ehrs'].cpu().numpy().astype(np.float16))


# save
with open((DATA_ROOT + '/shap_values_averaged_{}.pkl').format(chosen_def), 'wb') as f:
    pickle.dump({
        "shap_values_accumulated": shap_values_accumulated,
        "first_batch": first_batch
    }, f)

# python3 -m SHAP_test fall20nadir90


# SHAP plots

In [ ]:
import shap
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Use chosen_def from the generation cell above.
# load previous saved shap value
with open((DATA_ROOT + '/shap_values_averaged_{}.pkl').format(chosen_def), 'rb') as f:
    saved = pickle.load(f)

    # for those that is not averaged
    for i in range(3):
        saved['shap_values_accumulated'][i] = np.concatenate(saved['shap_values_accumulated'][i], axis=0)
    shap_values_averaged = saved['shap_values_accumulated']

    for k in saved['first_batch']:
        saved['first_batch'][k] = np.concatenate(saved['first_batch'][k], axis=0)
    first_batch = saved['first_batch']

# Historical analysis variant: unaggregated SHAP plots; adapt tensor access and EHR indices to the current saved format.
# # plot
# shap.summary_plot(
#     shap_values_averaged[0].reshape(shap_values_averaged[0].shape[0], -1), 
#     first_batch['bodies'].numpy().reshape(first_batch['bodies'].numpy().shape[0], -1),
#     max_display=10
# )
# shap.summary_plot( # Tabular data contributions
#     shap_values_averaged[1].reshape(shap_values_averaged[1].shape[0], -1), 
#     first_batch['summaries'].numpy().reshape(first_batch['summaries'].numpy().shape[0], -1)
# )
# shap.summary_plot( # EHR contributions
#     shap_values_averaged[3].reshape(shap_values_averaged[3].shape[0], -1), 
#     first_batch['ehrs'].numpy().reshape(first_batch['ehrs'].numpy().shape[0], -1)
# )


In [ ]:
# check for the real time variable (physiological and machine parameters)
def get_real_time_variables(shap_values):
    # shap_values_averaged: bn, 8, 64, 1
    result_shape_values = shap_values.sum(axis=2)
    return result_shape_values

real_time_var_shap = get_real_time_variables(shap_values_averaged[0])
real_time_first_batch = get_real_time_variables(first_batch['bodies'])

plt.rc('font', size=12)          # controls default text sizes
plt.rc('axes', titlesize=22)     # fontsize of the axes title
plt.rc('axes', labelsize=22)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=16)    # fontsize of the tick labels
plt.rc('ytick', labelsize=16)    # fontsize of the tick labels
plt.rc('legend', fontsize=18)    # legend fontsize
plt.rc('figure', titlesize=10)

shap.summary_plot(
    real_time_var_shap.reshape(real_time_var_shap.shape[0], -1), 
    real_time_first_batch.reshape(real_time_first_batch.shape[0], -1),
    max_display=20,
    title="Real Time Variables", 
    plot_size=(10, 6),
    feature_names=[
        "Systolic Blood Pressure [Body]",
        "Diastolic Blood Pressure [Body]",
        "Heart Rate [Body]",
        "Body Temperature [Body]",
        'Blood Flow [Machine]', 
        'Dialysate Flow [Machine]', 
        'Dialysate Temperature [Machine]', 
        'UFR [Machine]'
    ],
    alpha=0.5,
    show=False
)

plt.title("(a) Real Time Variables", fontsize='26')
plt.xlabel("SHAP value (impact on model output)", fontsize='24')
plt.xticks(fontsize=16)
plt.yticks(fontsize=22)

plt.savefig((FIGURE_ROOT + '/shap/real_time_vars_{}.pdf').format(chosen_def), format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
# check for the main variable (3 main modalities)
def get_main_variables(shap_values):
    # shap_values_averaged: [bn, 8, 64, 1] * 3
    result_shape_values = shap_values[0].reshape(shap_values[0].shape[0], -1).sum(axis=1, keepdims=True)
    result_shape_values = np.concatenate((result_shape_values, shap_values[1].reshape(shap_values[1].shape[0], -1).sum(axis=1, keepdims=True)), axis=1)
    result_shape_values = np.concatenate((result_shape_values, shap_values[2].reshape(shap_values[2].shape[0], -1).sum(axis=1, keepdims=True)), axis=1)
    return result_shape_values

main_var_shap = get_main_variables([shap_values_averaged[0], shap_values_averaged[1], shap_values_averaged[2]]) + 1e-6
main_first_batch = get_main_variables([first_batch['bodies'], first_batch['summaries'], first_batch['ehrs']]) + 1e-6

plt.clf()
shap.summary_plot(
    main_var_shap.reshape(main_var_shap.shape[0], -1), 
    main_first_batch.reshape(main_first_batch.shape[0], -1),
    max_display=20,
    title="Real Time Variables", 
    plot_size=(8.5, 6),
    feature_names=[
        "Real Time Data",
        "Historical Dialysis Summary",
        "Care Records"
    ],
    alpha=0.2,
    show=False
)

plt.title("(b) Three Main Modalities", fontsize='26')
plt.xlabel("SHAP value (impact on model output)", fontsize='24')
plt.xticks(fontsize=16)
plt.yticks(fontsize=22)

plt.savefig((FIGURE_ROOT + '/shap/main_vars_{}.pdf').format(chosen_def), format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
main_first_batch[:, -1].sum()


In [ ]:
# check for the real time variable (physiological and machine parameters)
def get_real_time_time_variables(shap_values):
    # shap_values_averaged: bn, 8, 64, 1
    result_shape_values = shap_values.sum(axis=1)
    return result_shape_values

real_time_time_var_shap = get_real_time_time_variables(shap_values_averaged[0])
real_time_time_first_batch = get_real_time_time_variables(first_batch['bodies'])

feature_names = sorted([t*SAMPLE_EVERY for t in range(first_batch['bodies'].shape[2])], reverse=True)
feature_names = ["{} mins".format(t) for t in feature_names]

shap.summary_plot(
    real_time_time_var_shap.reshape(real_time_var_shap.shape[0], -1), 
    real_time_time_first_batch.reshape(real_time_first_batch.shape[0], -1),
    max_display=15,
    title="Real Time Variables", 
    plot_size=(10, 6),
    feature_names=feature_names,
    alpha=0.5,
    show=False
)

plt.title("(c) Timestamp before last measurement.", fontsize='28')
plt.xlabel("SHAP value (impact on model output)", fontsize='24')
plt.xticks(fontsize=16)
plt.yticks(fontsize=22)

plt.savefig((FIGURE_ROOT + '/shap/real_time_time_vars_{}.pdf').format(chosen_def), format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
# check for the real time variable (physiological and machine parameters)
def get_summary_time_time_variables(shap_values):
    # shap_values_averaged: bn, 64, 18, 1
    result_shape_values = shap_values.sum(axis=2)
    return result_shape_values

summary_time_time_var_shap = get_summary_time_time_variables(shap_values_averaged[1])
summary_time_time_first_batch = get_summary_time_time_variables(first_batch['summaries'])

feature_names = sorted([t+1 for t in range(first_batch['summaries'].shape[1])], reverse=True)
feature_names = ["{}th".format(t) for t in feature_names]

shap.summary_plot(
    summary_time_time_var_shap.reshape(real_time_var_shap.shape[0], -1), 
    summary_time_time_first_batch.reshape(real_time_first_batch.shape[0], -1),
    max_display=20,
    title="Real Time Variables", 
    plot_size=(10, 6),
    feature_names=feature_names,
    alpha=0.5,
    show=False
)

plt.title("(d) Previous Dialysis Session", fontsize='28')
plt.xlabel("SHAP value (impact on model output)", fontsize='24')
plt.xticks(fontsize=16)
plt.yticks(fontsize=22)

plt.savefig((FIGURE_ROOT + '/shap/summary_time_time_vars_{}.pdf').format(chosen_def), format="pdf", bbox_inches="tight")
plt.show()


# Extract embedding

In [ ]:
# data to save
embeds = list()
labels = dict()

# iterate over data
data_loader = DataLoader(eval_dataset, batch_size=1024, shuffle=False, drop_last=True)
for data_pack in tqdm(data_loader):
    output = model(
        data_pack['bodies'].to(DEVICE),
        data_pack['summaries'].to(DEVICE),
        data_pack['decays'].to(DEVICE),
        data_pack['ehrs'].to(DEVICE),
        data_pack['ehr_decays'].to(DEVICE),
        embed_only=True
    )

    # update label
    for labek_key in ['fall20', 'fall30', 'nadir90', 'nadir100', "hemo", "kdoqi"]:
        if labels.get(labek_key) is None:
            labels[labek_key] = list()
        labels[labek_key] = labels[labek_key] + data_pack[labek_key].cpu().detach().numpy().tolist()

    # update embeds
    embeds = embeds + output.cpu().detach().numpy().astype(np.float16).tolist()

# check
embeds = np.array(embeds)
embeds.shape


In [ ]:
with open((DATA_ROOT + '/test_embeddings_fold_0.pkl'), 'wb') as f:
    pickle.dump({
        "embeds": embeds,
        "labels": labels
    }, f)


In [ ]:
# load stored embedding data
with open((DATA_ROOT + '/test_embeddings_fold_0.pkl'), 'rb') as f:
    saved = pickle.load(f)
    embeds = saved['embeds']
    labels = saved['labels']


In [ ]:
# init
sampled_num = 4000

# Analysis variant: sample a single IDH definition instead of the combined definition below; use the matching label mask.
# chosen_def = 'fall20'
# pos_idxs = np.random.choice([i for i in range(len(labels[chosen_def])) if labels[chosen_def][i]], sampled_num, replace=False)
# neg_idxs = np.random.choice([i for i in range(len(labels[chosen_def])) if not labels[chosen_def][i]], sampled_num, replace=False)

chosen_def = ['nadir90', 'fall20']
pos_idxs = np.random.choice([i for i in range(len(labels[chosen_def[0]])) if labels[chosen_def[0]][i] and labels[chosen_def[1]][i]], sampled_num, replace=False)
neg_idxs = np.random.choice([i for i in range(len(labels[chosen_def[0]])) if not (labels[chosen_def[0]][i] and labels[chosen_def[1]][i])], sampled_num, replace=False)

# sampling
pos_samples = embeds[pos_idxs, :].tolist()
neg_samples = embeds[neg_idxs, :].tolist()

# dimension reduction
X_embedded = TSNE(n_components=2, learning_rate='auto',
                  init='pca', perplexity=30).fit_transform(np.array(pos_samples+neg_samples))
X_embedded.shape


In [ ]:
plt.rc('font', size=42)          # controls default text sizes
plt.rc('axes', titlesize=32)     # fontsize of the axes title
plt.rc('axes', labelsize=32)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=12)    # fontsize of the tick labels
plt.rc('ytick', labelsize=12)    # fontsize of the tick labels
plt.rc('legend', fontsize=22)    # legend fontsize

plt.clf()
fig, ax = plt.subplots(figsize=(10,10))
plt.scatter(X_embedded[:len(pos_samples), 0], X_embedded[:len(pos_samples), 1], label="Fall20Nadir90", edgecolors='black', alpha=0.5, c='#FF6B6B')
plt.scatter(X_embedded[len(pos_samples):, 0], X_embedded[len(pos_samples):, 1], label="Normal State", edgecolors='black', alpha=0.5, c='#A2C8E6')
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.legend()
plt.savefig((FIGURE_ROOT + '/embeddings/{}_embeds.pdf').format(chosen_def), format="pdf", bbox_inches="tight")
plt.show()
